# Aronnax demonstrator
NEMO config must be compiled and deployed first. Then go in Aronnax home and install it by running:

```NEMO_BLD=/PATH/TO/NEMO/CONFIG/ pip install .```

## *1. Module import*

In [ ]:
# Import module -- Note the build path
import Aronnax

## *2. Single Run without restart*
##### Create a NEMO solver on 2 processes for a config named pyORCA2

In [ ]:
solver = Aronnax.NEMO(config='pyORCA2', nproc=2)

##### **pyORCA2.seg00** directory has been created and will contain Run results and control files when done

In [ ]:
# Show current NEMO state -- worker subprocesses have not started yet
print(solver)

In [ ]:
# Start NEMO workers -- note the new NEMO state
solver.start()
print(solver)

#### Advance time loop by 10 iterations

In [ ]:
# NEMO status will be printed at the end of run
solver.advance(ite=10)

#### Advance NEMO by 10 days
You can run ```top``` in a terminal during execution to see the 2 python processes running.

In [ ]:
solver.advance(days=10)

#### Get temperature fields
A field is a Python object linked to NEMO arrays. The field values can be automatically updated with NEMO (monitoring) or frozen when field is created (snapshot).

In [ ]:
# Register a monitoring and a snapshot fields for temperature
temp_monitor = solver.field('t')
temp_snap = solver.field('t', save=True)

In [ ]:
# Show a random cell from process 2 (rank=1)
rank = 1
print( "t0: T_monitor - T_snap(t0): ", temp_monitor[rank][40,50,0] - temp_snap[rank][40,50,0] )

# update NEMO
solver.advance(ite=10)

# compare fields change
print( "t1: T_monitor - T_snap(t0): ", temp_monitor[rank][40,50,0] - temp_snap[rank][40,50,0] )

#### Terminate run

In [ ]:
solver.finish()

#### Check content of **pyORCA2.seg00** directory

In [ ]:
!ls pyORCA2.seg00

## *3. Three-times Run with restart*
#### Create same NEMO solver on 2 processes for a config named pyORCA2

In [ ]:
solver = Aronnax.NEMO(config='pyORCA2', nproc=2, verbose=False)

##### Note that this run already exists. To avoid erasing precisous results, a different directory **pyORCA2.seg00_namesake00** has been created.

In [ ]:
# Advance NEMO by 10 days and terminate execution with restart files
solver.start()
solver.advance(days=10)
solver.finish(write_restart=True)

In [ ]:
# Check restart files in pyORCA2.seg00_namesake00 directory
!ls pyORCA2.seg00_namesake00/*restart*

#### Set new NEMO run in the direct continuity of this one
Note the new name of the run. Here, **pyORCA2.seg01** will restart on **pyORCA2.seg00.namesake00** run.

In [ ]:
solver.set_next_run()

In [ ]:
# Advance NEMO by 5 days and restart again on a third run
solver.start()
solver.advance(days=5)
solver.finish(write_restart=True)
# -- -- --
solver.set_next_run()
# -- -- --
solver.start()
solver.advance(days=5)
solver.finish()

Our three-time run is composed of three segments : **pyORCA2.seg00.namesake00** --> **pyORCA2.seg01** --> **pyORCA2.seg02**
You can show the simulation tree with command...**oopsie...it does not exist yet =D**

## *4. Run with direct restart, and fail*
Now we wish to start a simulation from **pyORCA2.seg01** and fork the simulation tree toward name **forked_ORCA2**

#### Create a NEMO solver on 2 processes for a config named forked_ORCA2, and specify starting run

In [ ]:
solver = Aronnax.NEMO(config="forked_ORCA2", nproc=2, verbose=False)
solver.run_from("pyORCA2.seg01")

#### Let us crash the simulation by imposing a unphysical temperature field

In [ ]:
# start NEMO - define a field linked with temperature array
solver.start()
temp = solver.field('t')

# Advance NEMO and store temperature from rank 0 and 1
solver.advance(days=1)
tr0 = temp[0]
tr1 = temp[1]

# Advance NEMO again 
solver.advance(days=2)

In [ ]:
# Impose temperature from day 1 multiplied by 2
temp[0] = tr0*2
temp[1] = tr1*2

# Advance again NEMO
solver.advance(months=1)

#### Note the NEMO state. Let's take a look in the NEMO log file

In [ ]:
!tail -n 20 ocean.output_0001

##### The Run directory name has changed. It is now labellised as **forked_ORCA2.seg00.FAILED** to be ignored with restart procedure.

In [ ]:
!ls -d forked*

##### The Aronnax Master process and associated objects are still alive, and we can re-rexecute this run again. 

In [ ]:
solver.start()
solver.advance() # one iteration by default
solver.finish()

##### Note that a namesake directory has been created to differentiate this run with the failed one.

## Wrapper is currently at early development stage - any feedbacks are welcome !